# pyskills.ipynb
> Functions for view/modifying ipynb file notebook cells. Each operation returns unified diffs showing what changed.
> 
> ## Ipynb file cell editing
> 
> Cell tools take an ipynb path and a cell id, e.g:
> 
>     cell_replace_lines('nb.ipynb', cell_id, 2, 3, 'replaced')
>     cell_insert_line('nb.ipynb', cell_id, 0, 'first line')
> 
> Use `summary_nb` for a one-line-per-cell overview of a large notebook, and `view_nb` to view the whole notebook (pass `only_errors=True` after running tests to jump straight to the cells that errored, with their tracebacks). Use `view_cell` to see a cell's source with line numbers before editing. Use `add_cell` to insert a new cell before/after an existing cell id, and `del_cells` to delete cells.
> 
> ## Line filtering
> 
> `cell_str_replace`, `cell_strs_replace`, and `cell_del_lines` support `re_filter` and `invert_filter` for targeting only lines matching (or not matching) a regex, like ex's `g//` and `g!//`. Combine with `start_line`/`end_line` to restrict to a region.

In [ ]:
#| default_exp ipynb

In [ ]:
#| export
import difflib,re
from pathlib import Path
from tempfile import TemporaryDirectory

from fastcore.meta import splice_sig
from fastcore.xtras import truncstr
from pyskills.edit import *

In [ ]:
from fastcore.test import test_eq,test_fail

In [ ]:
#| export
_cell_edit_doc = f"""
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages"""

In [ ]:
#| export
def _cell_edit(f, name=None):
    def wrapper(fname:str, id:str|list[str], *args, update_output:bool=False, **kw):
        nb = Notebook.open(fname)
        def _one(cid):
            cell = nb[cid]
            text = str(cell.outputs) if update_output else cell.source
            if not text: return f"error: Cell has no {'output' if update_output else 'source'}"
            try: new_text = f(text, *args, **kw)
            except ValueError as e: return f'error: {e}'
            if update_output: cell.outputs = ast.literal_eval(new_text)
            else: nb[cid] = new_text
            diff = '\n'.join(list(difflib.unified_diff(text.splitlines(), new_text.splitlines(), n=1, lineterm=''))[2:])
            return diff or 'none: No changes.'
        if isinstance(id, list) or id == 'all':
            if id == 'all': id = [c.id for c in nb.cells]
            res = [(cid, r) for cid in id if not (r := _one(cid)).startswith(('error:', 'none:'))]
        else: res = _one(id)
        nb.save()
        return res
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _cell_edit_doc
    return res

In [ ]:
#| export
from fastcore.nbio import *

In [ ]:
_tmp = TemporaryDirectory()
_test_content = 'alpha\nbeta\ngamma\ndelta\n'

_nb_path = f'{_tmp.name}/test.ipynb'
_tnb = Notebook(new_nb([_test_content, 'other cell']))
_tnb.save(_nb_path)
_cid, _oid = _tnb[0].id, _tnb[1].id
def _nb_src(): return Notebook.open(_nb_path)[_cid].source
_cid, _oid

('f64d27aa', 'cdb55bbc')

In [ ]:
#| export
cell_insert_line = _cell_edit(insert_line, 'cell_insert_line')
cell_str_replace = _cell_edit(str_replace, 'cell_str_replace')
cell_strs_replace = _cell_edit(strs_replace, 'cell_strs_replace')
cell_replace_lines = _cell_edit(replace_lines, 'cell_replace_lines')
cell_del_lines = _cell_edit(del_lines, 'cell_del_lines')

In [ ]:
res = cell_insert_line(_nb_path, _cid, 0, 'first')
test_eq(_nb_src().splitlines()[0], 'first')
print(res)

@@ -1 +1,2 @@
+first
 alpha


In [ ]:
res = cell_str_replace(_nb_path, _cid, 'beta', 'BETA')
assert 'BETA' in _nb_src(), f"Expected 'BETA' in cell"
print(res)

@@ -2,3 +2,3 @@
 alpha
-beta
+BETA
 gamma


In [ ]:
res = cell_strs_replace(_nb_path, _cid, ['gamma', 'delta'], ['GAMMA', 'DELTA'])
test_eq(_nb_src().splitlines()[-2:], ['GAMMA', 'DELTA'])
print(res)

@@ -3,3 +3,3 @@
 BETA
-gamma
-delta
+GAMMA
+DELTA


In [ ]:
res = cell_replace_lines(_nb_path, _cid, 2, 3, 'two\nthree\n')
test_eq(_nb_src().splitlines()[1:3], ['two', 'three'])
print(res)

@@ -1,4 +1,4 @@
 first
-alpha
-BETA
+two
+three
 GAMMA


In [ ]:
res = cell_del_lines(_nb_path, _cid, 1)
test_eq(_nb_src().splitlines()[0], 'two')
print(res)

@@ -1,2 +1 @@
-first
 two


In [ ]:
#| export
def add_cell(fname:str, # ipynb to edit
             source:str, # source for the new cell
             cell_type:str='code', # 'code', 'markdown', or 'raw'
             before:str=None, # id of cell to insert before
             after:str=None): # id of cell to insert after
    "Add a new cell before/after an existing cell (pass exactly one), returning the new cell's id"
    if (before is None)==(after is None): raise ValueError('Pass exactly one of `before` or `after`')
    nb = Notebook.open(fname)
    idx = nb.cells.index(nb[before or after]) + (after is not None)
    cell = mk_cell(source, cell_type)
    nb.cells.insert(idx, cell)
    nb.save()
    return cell.id

`add_cell` inserts a whole new cell (rather than editing within one), placed relative to an existing cell id. It returns the new cell's id so follow-up edits can target it.

In [ ]:
_nid = add_cell(_nb_path, 'zeta', after=_cid)
test_eq([c.id for c in Notebook.open(_nb_path).cells[:2]], [_cid, _nid])
test_eq(Notebook.open(_nb_path)[_nid].source, 'zeta')

_mid = add_cell(_nb_path, '# note', cell_type='markdown', before=_cid)
test_eq((Notebook.open(_nb_path)[0].id, Notebook.open(_nb_path)[0].cell_type), (_mid, 'markdown'))

test_fail(add_cell, args=(_nb_path, 'x'), exc=ValueError)
test_fail(add_cell, kwargs=dict(fname=_nb_path, source='x', before=_cid, after=_nid), exc=ValueError)

In [ ]:
#| export
def del_cells(fname:str, # ipynb to edit
              *ids:str): # ids of cells to delete
    "Delete cells by id"
    nb = Notebook.open(fname)
    for i in ids: del nb[i]
    nb.save()

`del_cells` removes whole cells. A missing id raises `KeyError`, and nothing is saved in that case.

In [ ]:
del_cells(_nb_path, _nid, _mid)
assert _nid not in Notebook.open(_nb_path) and _mid not in Notebook.open(_nb_path)
test_fail(del_cells, args=(_nb_path, 'nonexistent'), exc=KeyError)

## Moving cells

`copy_cells`/`cut_cells` copy one or more cells (by id) into a small paste buffer; `paste_cells` inserts the buffered cells before/after a cell id in any notebook -- including a different file or project. To move cells across notebooks: `cut_cells` from the source, then `paste_cells` into the destination.

In [ ]:
#| export
_paste_buf = []

def copy_cells(fname:str, # ipynb to copy from
               *ids:str): # ids of cells to copy
    "Copy cells into the paste buffer (replacing its contents), for later `paste_cells`"
    nb = Notebook.open(fname)
    global _paste_buf
    _paste_buf = [(nb[i].cell_type, nb[i].source) for i in ids]


`copy_cells` reads cells into the buffer without touching the source notebook. Like `del_cells`, a missing id raises `KeyError`.

In [ ]:
copy_cells(_nb_path, _oid)
test_eq(_paste_buf, [('code', 'other cell')])
assert _oid in Notebook.open(_nb_path)  # source untouched

test_fail(copy_cells, args=(_nb_path, 'nonexistent'), exc=KeyError)

In [ ]:
#| export
def cut_cells(fname:str, # ipynb to cut from
              *ids:str): # ids of cells to cut
    "Copy cells into the paste buffer, then delete them from `fname`"
    copy_cells(fname, *ids)
    del_cells(fname, *ids)


`cut_cells` is `copy_cells` followed by `del_cells` -- same `KeyError`-on-missing-id behavior, but the source cells are gone afterwards.

In [ ]:
cut_cells(_nb_path, _oid)
test_eq(_paste_buf, [('code', 'other cell')])
assert _oid not in Notebook.open(_nb_path)

In [ ]:
#| export
def paste_cells(fname:str, # ipynb to paste into
                before:str=None, # id of cell to insert before
                after:str=None): # id of cell to insert after
    "Insert the buffered cells (from `copy_cells`/`cut_cells`) before/after a cell id, returning the new ids"
    if not _paste_buf: raise ValueError('Paste buffer is empty -- use `copy_cells`/`cut_cells` first')
    if (before is None)==(after is None): raise ValueError('Pass exactly one of `before` or `after`')
    ids,anchor,is_after = [],(after if after is not None else before),(after is not None)
    for cell_type,source in _paste_buf:
        nid = add_cell(fname, source, cell_type, **({'after':anchor} if is_after else {'before':anchor}))
        ids.append(nid); anchor,is_after = nid,True
    return ids

`paste_cells` inserts the buffered cells before/after a cell id -- in the same notebook, a different notebook, or a different project, since `fname` is just a path. Pasting doesn't clear the buffer, so the same copy can be pasted into several places, and multiple buffered cells keep their relative order.

In [ ]:
_dst_path = f'{_tmp.name}/test2.ipynb'
Notebook(new_nb(['dest cell'])).save(_dst_path)
_did = Notebook.open(_dst_path)[0].id

new_ids = paste_cells(_dst_path, after=_did)
test_eq(Notebook.open(_dst_path)[new_ids[0]].source, 'other cell')

paste_cells(_dst_path, before=_did)  # buffer still holds it -- can paste again
test_eq(Notebook.open(_dst_path)[0].source, 'other cell')

In [ ]:
_ord_path = f'{_tmp.name}/ord.ipynb'
Notebook(new_nb(['a', 'b', 'c'])).save(_ord_path)
_a,_b,_c = [c.id for c in Notebook.open(_ord_path).cells]

copy_cells(_ord_path, _a, _b)
paste_cells(_ord_path, after=_c)
test_eq([c.source for c in Notebook.open(_ord_path).cells], ['a', 'b', 'c', 'a', 'b'])  # order preserved

test_fail(paste_cells, args=(_dst_path,), exc=ValueError)  # neither before nor after
test_fail(paste_cells, kwargs=dict(fname=_dst_path, before=_did, after=new_ids[0]), exc=ValueError)  # both

## Viewing

In [ ]:
#| export
def view_cell(fname:str, id:str, nums:bool=True):
    "Show cell source with optional line numbers"
    return Notebook.open(fname).view(id, nums=nums)

`view_cell` displays a cell's source with line numbers, useful for checking content before editing:

In [ ]:
print(view_cell(_nb_path, _cid))

     1 │ two


     2 │ three


     3 │ GAMMA


     4 │ DELTA


In [ ]:
#| export
def view_nb(fname:str, incl_out:bool=False, only_errors:bool=False):
    "Show notebook source as concise xml; `incl_out` includes output, `only_errors` shows only cells with an error output (implies output included)"
    nb = Notebook.open(fname)
    if only_errors:
        cells = [c for c in nb.cells if any(o.get('output_type')=='error' for o in c.get('outputs',[]))]
        return cells2xml(cells, path=nb.path.name, incl_out=True)
    return repr(nb) if incl_out else nb.concise


In [ ]:
view_nb(_nb_path)

'<nb path="test.ipynb"><code id="fc4d7e8e">two\nthree\nGAMMA\nDELTA</code><code id="b0e7ba69">other cell</code></nb>'

In [ ]:
_err_nb = f'{_tmp.name}/err.ipynb'
_enb = Notebook(new_nb(['ok = 1', 'boom']))
_ecid,_bid = _enb[0].id, _enb[1].id
_enb[_bid]['outputs'] = [mk_error(['Traceback (most recent call last):', 'ValueError: boom'], ename='ValueError', evalue='boom')]
_enb.save(_err_nb)

res = view_nb(_err_nb, only_errors=True)
assert _bid in res and _ecid not in res
assert 'ValueError' in res

In [ ]:
#| export
def summary_nb(fname:str,      # ipynb to summarize
               maxlen:int=120): # truncate each cell's source to this
    "One line per cell: id, type, and truncated/escaped source"
    def _l(c): return f"{c.id}:{c.cell_type[0]}:{truncstr(c.source.replace(chr(10), r'\n'), maxlen)}"
    return '\n'.join(_l(c) for c in Notebook.open(fname).cells)

`summary_nb` gives a skimmable overview - one truncated line per cell, like `nbrg`'s output but for every cell. Use it to survey a large notebook before drilling in with `view_cell` or `view_nb`.

In [ ]:
res = summary_nb(_nb_path)
print(res)
lines = res.splitlines()
test_eq(len(lines), len(Notebook.open(_nb_path).cells))
test_eq(lines[0].split(':')[:2], [_cid, 'c'])

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()